# **Tutorial:** Introduction to Large Language Models (LLMs) with HuggingFace

## Objective:
The goal of this workshop is to introduce students to the basic concepts of large language models (LLMs) and give them hands-on experience using Hugging Face libraries to work with pretrained natural language models.

## Contents:
1. Introduction to Large Language Models (LLMs)
2. Environment setup
3. Practical examples
4. Exercise

## 1. Introduction to Large Language Models (LLMs)

Large language models (LLMs) are AI models trained on huge amounts of text to understand and generate natural language. These models can perform a variety of natural language processing (NLP) tasks such as translation, summarization, text generation, question answering, and more.

## 2. Environment setup

For this workshop, we'll use the Hugging Face `transformers` library. First, we need to install the required packages.

> **Note (2026):** Hugging Face is still relevant. Instead of `gpt2`, we now use small, lightweight *instruct* models (`Qwen2.5-0.5B-Instruct`, `SmolLM2-360M-Instruct`). Tomorrow we'll see larger models via API (OpenAI, DeepSeek).

In [ ]:
# Install the transformers and torch libraries
%pip install transformers
%pip install torch
%pip install tf-keras

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_10_13_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2025.11.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-macosx_10_12_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-macosx_10_12_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-macosx_10_12_x86_64.whl.metadata (4.9 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached hf_xet-1.2.0-cp37-abi3-macosx_10_12_x86_64.whl (2.9 MB)
Using cached tokenizers-0.22.1-cp39-abi3-macosx_10_12_x86_64

We import `pipeline` from the `transformers` library

In [ ]:
from transformers import pipeline

/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Practical examples of loading and using pretrained models

### 3.1 Text generation (`text-generation`)

Let's load a pretrained Hugging Face model and use it to generate text.

In [ ]:
# Load a text-generation pipeline with a small, current instruct model
generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B-Instruct')

# Instruct-tuned models expect a "messages" format (like a chat)
messages = [
    {"role": "user", "content": "Once upon a time there was a kid who had a bike. Continue the story in 2 lines."}
]

text = generator(
    messages,
    max_new_tokens=80, # Maximum length of the generated text.
    num_return_sequences=1,
    temperature=0.7, # This parameter controls the randomness of the responses. Lower values make the model more predictable, while higher values make it more creative.
    )

# The model's reply is the last generated message
print(text[0]['generated_text'][-1]['content'])

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Secuencia 1: Había una vez, un niño que tenía una bicicleta, una español, español una de la mía y algunos de de los de llamos,

Secuencia 2: Había una vez, un niño que tenía una bicicleta para el puede.

Tengo de la tengo, no a las casas de cualquier en otr

Secuencia 3: Había una vez, un niño que tenía una bicicleta.

"We know that, though we have a strong moral position, we have no understanding of what constitutes a moral position. We have been



Now let's use the model to generate text from several prompts

In [ ]:
# Example of text generation with different prompts
prompts = [
    "In the future, AI will",
    "The secret to happiness is",
    "The quick brown fox"
]

for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    text = generator(messages, max_new_tokens=60, num_return_sequences=1)
    print(f"Prompt: {prompt}")
    print(f"Generated Text: {text[0]['generated_text'][-1]['content']}\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: In the future, AI will
Generated Text: In the future, AI will become part of every decision, and will be able to be used for decisions on issues like global warming, the need for transparency of regulatory decisions, etc.

There are also big differences between AI and regular people (



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: The secret to happiness is
Generated Text: The secret to happiness is to go to your most peaceful place, where you are aware of that freedom without making mistakes and no one will look at you like you are not there, so make no mistake and believe that you have made a good decision.

Prompt: The quick brown fox
Generated Text: The quick brown fox at your left and the short, long-haired blonde at your right. They both talk quickly in the same silent voice.

This isn't your fault, I thought I'd ask.

"We have to hurry



> **Note:** *gated* models (Llama, Mistral) require `huggingface-cli login`. The ones we use here are public.

### 3.2 Question answering (`question-answering`)

We can use a question-answering pipeline to find answers within a given context.

In [ ]:
# Load a question-answering pipeline
qa_pipeline = pipeline('question-answering')

# Define the context and the question
#context = "You are a mathematics teacher that can explain complex concepts in simple terms."
context = (
    "Artificial intelligence (AI) is a branch of computer science that aims to create machines that can perform tasks that would normally require human intelligence. "
    "Machine learning (ML) is a subset of AI that involves the use of algorithms and statistical models to enable computers to improve their performance on a task through experience. "
    "One common application of ML is in the field of natural language processing (NLP), where algorithms are used to understand and generate human language. "
    "For example, GPT-4o is a state-of-the-art language model developed by OpenAI that can generate human-like text based on a given prompt."
)
question = "What is machine learning?"

# Get the answer
result = qa_pipeline(question=question, context=context)
print(f"Question: {question}")
print(f"Answer: {result['answer']}")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


Pregunta: What is machine learning?
Respuesta: a subset of AI that involves the use of algorithms and statistical models


### 3.3 Text summarization (`summarization`)

Use a pretrained model to generate a summary of a long text.

In [ ]:
# Load a summarization pipeline
summarizer = pipeline('summarization')

# Long text to summarize
long_text = (
"Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term can also apply to any machine that exhibits traits associated with a human mind, such as learning and problem-solving. The ideal characteristic of artificial intelligence is its ability to rationalize and take actions that have the best chance of achieving a specific goal. A subset of artificial intelligence is machine learning, which refers to the idea that computer systems can learn from data, identify patterns, and make decisions with minimal human intervention."
)

# Generate the summary
summary = summarizer(long_text, max_length=50, min_length=25)
print("Summary:")
print(summary[0]['summary_text'])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


Resumen:
 La inteligencia artificial (IA) se refiere a la simulación of la inteligenia humana . La característica ideal de la IAI is su capacidad for racionalizar y tomar


### 3.4 Text translation (`translation`)

We use a pretrained English-to-Spanish translation model

In [ ]:
%pip install sentencepiece
import sentencepiece

# Load a translation pipeline, specifying the model
translation_pipeline = pipeline('translation', model='Helsinki-NLP/opus-mt-en-es')

# Define the text to translate
text = "Persistent homology is a method for computing topological features of a space at different spatial resolutions. More persistent features are detected over a wide range of spatial scales and are deemed more likely to represent true features of the underlying space rather than artifacts of sampling, noise, or particular choice of parameters"

# Get the translation
result = translation_pipeline(text)
print(f"Original text: {text}")
print(f"Translation: {result[0]['translation_text']}")

Note: you may need to restart the kernel to use updated packages.


/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Texto original: Persistent homology is a method for computing topological features of a space at different spatial resolutions. More persistent features are detected over a wide range of spatial scales and are deemed more likely to represent true features of the underlying space rather than artifacts of sampling, noise, or particular choice of parameters
Traducción: La homología persistente es un método para calcular las características topológicas de un espacio en diferentes resoluciones espaciales. Se detectan características más persistentes en una amplia gama de escalas espaciales y se considera más probable que representen características verdaderas del espacio subyacente en lugar de artefactos de muestreo, ruido o elección particular de parámetros.


### 3.4 Sentiment analysis (`sentiment-analysis`)

Now we'll use a sentiment analysis model to predict whether a text is positive or negative.

In [ ]:
# Load the sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Text to analyze sentiment
text = "The price of the NVIDIA stock will increase tomorrow."
#text = "Today I have a quiz for the course Artificial Intelligence."
#text = "A cat is running in green fields in the summer."

# Perform sentiment analysis
sentiment_result = sentiment_analyzer(text)

# Print the sentiment result
print("Sentiment Analysis Result:")
for result in sentiment_result:
    print(f"Label: {result['label']}, Score: {result['score']}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Sentiment Analysis Result:
Label: POSITIVE, Score: 0.9853973984718323


### 3.5 Asking questions to a table (`table-question-answering`)

Now let's ask questions to a Pandas DataFrame

In [19]:
import pandas as pd

# Load a table-question-answering pipeline
#table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
table_qa = pipeline("table-question-answering")

#Create a table as a pandas DataFrame
table = {
    "Name": ["Karen", "Anna", "Pedro"],
    "Age": ["22", "31", "19"],  # Ensure all entries are strings
    "Interest": ["Cybersecurity", "Numerical Analysis", "Functional Analysis"]
}

# Question about the table
#question= "What is the interest of Karen?"
#question = "What is the age of Pedro?"
question = "What is the average age of the table?" # 22 + 31 + 19 = 72 / 3 = 24

# Generate the answer
answer = table_qa(table=table, query=question)

# Print the question and the answer
print("Question:", question)
print("Answer:", answer['answer'])

No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision 69ceee2 (https://huggingface.co/google/tapas-base-finetuned-wtq).
Using a pipeline without specifying a model name and revision in production is not recommended.


Question: What is the average age of the table?
Answer: AVERAGE > 22, 31, 19


## Other tasks:

- 'audio-classification'
- 'automatic-speech-recognition'
- 'conversational'
- 'depth-estimation'
- 'document-question-answering'
- 'feature-extraction'
- 'fill-mask'
- 'image-classification'
- 'image-feature-extraction'
- 'image-segmentation'
- 'image-to-image'
- 'image-to-text'
- 'mask-generation'
- 'ner', 'object-detection'
- 'question-answering'
- 'sentiment-analysis'
- 'summarization'
- 'table-question-answering'
- 'text-classification'
- 'text-generation'
- 'text-to-audio'
- 'text-to-speech'
- 'text2text-generation'
- 'token-classification'
- 'translation'
- 'video-classification'
- 'visual-question-answering'
- 'vqa'
- 'zero-shot-audio-classification'
- 'zero-shot-classification'
- 'zero-shot-image-classification'
- 'zero-shot-object-detection'
- 'translation_XX_to_YY'

### **Individual Activity (30 min):** Comparing Recent Hugging Face Models

Pick a task (`text-generation`, `zero-shot-classification`, `summarization`, `sentiment-analysis`, etc.) and two recent, lightweight models different from the ones used here. Suggestions: `Qwen/Qwen2.5-1.5B-Instruct`, `HuggingFaceTB/SmolLM2-360M-Instruct`, `microsoft/Phi-3-mini-4k-instruct`, or explore [Trending on Hugging Face](https://huggingface.co/models).

Try both with the same input and compare: which one answered better or faster? Save the notebook as a PDF and upload it to Moodle.

> Tomorrow: bigger models (OpenAI, DeepSeek) via **API**.

*Estimated time:* 30 min

In [ ]:
# Write your code below



**Save this Notebook as a PDF and upload it to the designated activity on Moodle.**  

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>